# BS-Detector: birth scars, poles and growth curves from a time-lapse movie

Runs the pipeline on one brightfield movie, then lets you **check and correct** it:

1. **Segment** every frame with Cellpose-SAM (the slow step; uses Colab's GPU).
2. **Track** cells, assign old and new poles, find each cell's birth scar.
3. **Growth curves** for every cell born and dividing in the movie.
4. **Review**: which cells have frames worth checking, and why.
5. **Look at a cell**: where the detector put its scar and poles, frame by frame.
6. **Fix** what's wrong: click the scar, exclude frames, swap the ends, or enter your own lengths.
   Fiji works too (see step 6).

**Before you start:** *Runtime > Change runtime type > T4 GPU*. For your own movie, copy it
(`.nd2`, SlideBook `.sldy` + `.dir`, `.tif` or `.h5`) into Google Drive.

**What gets saved:** for your own movie, the results folder in Drive gets the CSV tables and a
small `working/` folder of cell outlines (so a disconnected run resumes, and cells can be
checked later). Figures are only shown here unless you tick `SAVE_FIGURES`. **Leave `MOVIE` empty
to try the 5-frame sample:** nothing is written to your Drive. The sample is too short for
growth curves, but every other step works.

**If Colab disconnects,** run all cells again: segmentation continues where it stopped.
Guide: [docs/USER_GUIDE.md](https://github.com/widenerm/pombe-bs-detector/blob/main/docs/USER_GUIDE.md)

In [ ]:
#@title 1. Install BS-Detector (about 2 minutes)
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU, then run this cell again"
!pip install -q "bs-detector @ git+https://github.com/widenerm/pombe-bs-detector"
!bsd --help | head -3

In [ ]:
#@title 2. Choose the movie
from pathlib import Path

MOVIE = ""  #@param {type:"string"}
#@markdown Path in Drive, e.g. `/content/drive/MyDrive/movies/11_19_2025_rgf1DEL_gpd1DEL_YE_.nd2`. Empty = sample movie (nothing saved to Drive).
OUT = ""  #@param {type:"string"}
#@markdown Results folder. Empty = `MyDrive/bsd_runs/<movie name>`.
POSITION = ""  #@param {type:"string"}
#@markdown Only for multi-position files; empty = every position, each in its own folder.
HAND = ""  #@param {type:"string"}
#@markdown Optional: hand-tracked growth workbook (.xlsx) to plot next to the pipeline.
HAND_TYPE = ""  #@param {type:"string"}
#@markdown Which cells in the workbook, e.g. `y_wt` (YE wild type) or `y_rgf1delgpd1del`.
SAVE_FIGURES = False  #@param {type:"boolean"}
#@markdown Also save the figures (PNG/PDF) to `OUT/figures` in Drive.

if MOVIE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = OUT or f'/content/drive/MyDrive/bsd_runs/{Path(MOVIE).stem}'
else:
    MOVIE = '/content/sample.h5'
    !wget -q -O {MOVIE} https://github.com/widenerm/pombe-bs-detector/raw/main/data/sample/sample.h5
    OUT = '/content/bsd_runs/sample'          # on this Colab machine only
FIGS = '/content/bsd_figures'                 # figures stay on this machine
FRAMES = '/content/bsd_frames'                # a resampled copy of the movie, if one is needed
pos = f'--position "{POSITION}"' if POSITION else ''
hand = (f'--hand "{HAND}"' + (f' --hand-type "{HAND_TYPE}"' if HAND_TYPE else '')) if HAND else ''
print('movie:  ', MOVIE); print('results:', OUT)
!bsd info "{MOVIE}" {pos}

In [ ]:
#@title 3. Segment (GPU; resumable)
!bsd segment "{MOVIE}" --out "{OUT}" {pos} --frames-dir "{FRAMES}"

In [ ]:
#@title 4. Track cells, assign poles, find birth scars; growth curves
!bsd track "{OUT}" --figures-dir "{FIGS}"
!bsd growth "{OUT}" {hand} --figures-dir "{FIGS}"

import shutil, pandas as pd
from IPython.display import Image, Markdown, display
RUNS = [Path(OUT)] if (Path(OUT) / 'run.json').exists() else sorted(p.parent for p in Path(OUT).glob('*/run.json'))
for run in RUNS:
    figs = Path(FIGS) / run.name
    display(Markdown(f'## {run.name}'))
    m = pd.read_csv(run / 'measurements.csv')
    display(Markdown(f"{m.cell_name.nunique()} cells, {len(m)} cell-frames"))
    for png in sorted(figs.glob('qc_frame_*.png'))[:2] + sorted(figs.glob('growth_since_birth*.png')):
        display(Image(str(png), width=700))
    if (run / 'growth' / 'growth_curves_summary.csv').exists():
        display(pd.read_csv(run / 'growth' / 'growth_curves_summary.csv'))
    if SAVE_FIGURES and figs.exists():
        shutil.copytree(figs, run / 'figures', dirs_exist_ok=True)
        print('figures saved to', run / 'figures')
RUN = str(RUNS[0])
print('Tables are in', OUT)

## Check and correct

`measurements.csv` is the detector's output and is never changed. Your fixes go into
`corrections/edits.csv` in the results folder (you can also edit that file by hand), and the next
growth step writes `measurements_corrected.csv` with a `corrected` column saying what changed.

**Fiji instead:** open the movie in Fiji, run
[`fiji/BS_check_cell.ijm`](https://github.com/widenerm/pombe-bs-detector/blob/main/fiji/BS_check_cell.ijm),
type the cell's name (e.g. `A01`), click the scar on each frame, and save the CSV in the results
folder's `corrections/`. Then run step 7.

In [ ]:
#@title 5. Which cells to check
#@markdown Cells in the growth curves come first. `why` counts the flagged frames by reason:
#@markdown **unsure** (detector's own flag), **jump** (a length >0.5 µm off the cell's running median: the "crazy numbers"),
#@markdown **off-axis** (scar far from the cell's midline), **interpolated**, **no scar**, **outline** (segmentation flag).
#@markdown `x`, `y`, `at_frame` say where to find the cell (movie pixels; Fiji slice = frame + 1).
WHICH_POSITION = ""  #@param {type:"string"}
#@markdown Multi-position movies only: the position folder to check, e.g. `position_2` (empty = the first).
if WHICH_POSITION:
    RUN = str(Path(OUT) / WHICH_POSITION)
from pombe_tracker import colab
print('checking', RUN)
display(colab.review(RUN, top=25))

In [ ]:
#@title 6. Look at a cell, and fix it
CELL = "A0"  #@param {type:"string"}
FRAMES = ""  #@param {type:"string"}
#@markdown e.g. `52` or `40-60`; empty = the frames to check (or evenly spaced).
ACTION = "just look"  #@param ["just look", "click the scar", "exclude these frames", "swap new/old ends", "mark checked (detector is right)", "enter my own end lengths"]
NEW_END_UM = 0.0  #@param {type:"number"}
OLD_END_UM = 0.0  #@param {type:"number"}
NOTE = ""  #@param {type:"string"}
from pombe_tracker.corrections import add_edit
from pombe_tracker.workflow import Run

run = Run(RUN)
if ACTION == "just look":
    colab.show(RUN, CELL, FRAMES or None)
elif ACTION == "click the scar":
    colab.place_scars(RUN, CELL, FRAMES)
else:
    fields = {"exclude these frames": dict(exclude="yes"), "swap new/old ends": dict(swap_poles="yes"),
              "mark checked (detector is right)": dict(checked="yes"),
              "enter my own end lengths": dict(new_end_length_um=NEW_END_UM, old_end_length_um=OLD_END_UM)}[ACTION]
    add_edit(run, CELL, FRAMES, note=NOTE, **fields)
    print(f'recorded for {CELL}, frames {FRAMES or "all"}: {ACTION}')
print('corrections so far:')
display(pd.read_csv(run.corrections_dir / 'edits.csv'))

In [ ]:
#@title 7. Apply the corrections and redraw growth curves
!bsd growth "{RUN}" {hand} --figures-dir "{FIGS}"
!bsd review "{RUN}" --top 10
for png in sorted((Path(FIGS) / Path(RUN).name).glob('growth_since_birth*.png')):
    display(Image(str(png), width=700))